# HTTPS用CAと証明書の準備

ルートCAを作成し、各ホストのHTTPS証明書を発行・配置する。D10で作成したinventoryとgroup_varsを使用する。操作端末にはOpenSSLとkeytoolが必要。


## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*


# 対象クラスタの設定


In [ ]:
target_group = 'hadoop_all_testcluster'
target_group


In [ ]:
import os
import yaml

with open('group_vars/' + target_group) as stream:
    configuration = yaml.safe_load(stream)


inventory上の接続先と、証明書に記載するホスト名を指定する。接続先のホスト名を確認する。


In [ ]:
!ansible {target_group} -a 'hostname -f'


In [ ]:
certificate_hosts = {
    'xxx.xxx.xxx.64': 'test01',
    'xxx.xxx.xxx.65': 'test02',
}
certificate_hosts


# CAの準備

CAの秘密鍵と発行した証明書を保管するディレクトリを指定する。秘密鍵は暗号化し、このディレクトリへのアクセスを制限する。


In [ ]:
pki_dir = os.path.expanduser('~/.local/share/hadoop-pki/initial')
pki_dir


In [ ]:
ca_subject = '/CN=Hadoop Root CA'
ca_days = 3650
certificate_days = 365
(ca_subject, ca_days, certificate_days)


In [ ]:
ca_key_file = os.path.join(pki_dir, 'ca.key')
ca_certificate_file = os.path.join(pki_dir, 'ca.crt')
truststore_file = os.path.join(pki_dir, 'truststore.jks')
(ca_key_file, ca_certificate_file, truststore_file)


In [ ]:
!openssl version
!keytool -help


CA秘密鍵のパスワードを入力する。ホストのkeystoreとtruststoreには、group_varsに指定したパスワードを使用する。


In [ ]:
import getpass

os.environ['HADOOP_CA_PASSWORD'] = getpass.getpass('CA key password: ')
os.environ['HADOOP_KEYSTORE_PASSWORD'] = configuration['https_keystore_pass']
os.environ['HADOOP_PRIVATE_KEY_PASSWORD'] = configuration['https_privkey_pass']
os.environ['HADOOP_TRUSTSTORE_PASSWORD'] = configuration['https_truststore_pass']


In [ ]:
!install -d -m 0700 "{pki_dir}"
!test ! -e "{ca_key_file}"
!test ! -e "{ca_certificate_file}"


ルートCAの鍵と自己署名証明書を作成する。


In [ ]:
!umask 077; openssl req -new -x509 -newkey rsa:3072 -sha256 -days {ca_days} \
    -subj "{ca_subject}" \
    -addext "basicConstraints=critical,CA:TRUE,pathlen:0" \
    -addext "keyUsage=critical,keyCertSign,cRLSign" \
    -passout env:HADOOP_CA_PASSWORD \
    -keyout "{ca_key_file}" \
    -out "{ca_certificate_file}"


CAのSubject、有効期限、CAとしての制約とフィンガープリントを確認する。


In [ ]:
!openssl x509 -in "{ca_certificate_file}" -noout -text -fingerprint -sha256


# ホスト証明書の発行

ホストごとに新しい鍵とCSRを作成する。ホスト名はSubject Alternative Nameに記載する。


In [ ]:
for address, hostname in certificate_hosts.items():
    !umask 077; keytool -genkeypair -alias server -keyalg RSA -keysize 3072 \
        -dname "CN={hostname}" -ext "SAN=dns:{hostname}" -storetype JKS \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD \
        -keypass:env HADOOP_PRIVATE_KEY_PASSWORD


In [ ]:
for address, hostname in certificate_hosts.items():
    !keytool -certreq -alias server -ext "SAN=dns:{hostname}" \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD \
        -keypass:env HADOOP_PRIVATE_KEY_PASSWORD -file "{pki_dir}/{hostname}.csr"


In [ ]:
for address, hostname in certificate_hosts.items():
    !openssl req -in "{pki_dir}/{hostname}.csr" -noout -text -verify


発行する証明書の用途とホスト名を指定する。


In [ ]:
for address, hostname in certificate_hosts.items():
    extension_file = os.path.join(pki_dir, hostname + '.ext')
    with open(extension_file, 'w') as stream:
        stream.write('basicConstraints=critical,CA:FALSE\nkeyUsage=critical,digitalSignature,keyEncipherment\nextendedKeyUsage=serverAuth\nsubjectAltName=DNS:' + hostname + '\n')
    !cat "{extension_file}"


In [ ]:
for address, hostname in certificate_hosts.items():
    !openssl x509 -req -in "{pki_dir}/{hostname}.csr" -CA "{ca_certificate_file}" -CAkey "{ca_key_file}" \
        -passin env:HADOOP_CA_PASSWORD -CAcreateserial -days {certificate_days} -sha256 \
        -extfile "{pki_dir}/{hostname}.ext" \
        -out "{pki_dir}/{hostname}.crt"


発行元、有効期限、ホスト名を確認し、CA証明書で検証する。


In [ ]:
for address, hostname in certificate_hosts.items():
    !openssl x509 -in "{pki_dir}/{hostname}.crt" -noout -text -fingerprint -sha256
    !openssl verify -CAfile "{ca_certificate_file}" -verify_hostname "{hostname}" "{pki_dir}/{hostname}.crt"


発行した証明書を、鍵を作成したkeystoreへ取り込む。


In [ ]:
for address, hostname in certificate_hosts.items():
    !keytool -importcert -noprompt -alias root-ca -file "{ca_certificate_file}" \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD
    !keytool -importcert -noprompt -alias server -file "{pki_dir}/{hostname}.crt" \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD \
        -keypass:env HADOOP_PRIVATE_KEY_PASSWORD


In [ ]:
for address, hostname in certificate_hosts.items():
    !keytool -list -v -keystore "{pki_dir}/{hostname}-keystore.jks" -storepass:env HADOOP_KEYSTORE_PASSWORD


# 信頼するCAの設定

全ホストで、このCAを信頼するtruststoreを使用する。


In [ ]:
!keytool -importcert -noprompt -alias root-ca -file "{ca_certificate_file}" \
    -keystore "{truststore_file}" -storetype JKS \
    -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -list -v -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


# 証明書の配置

group_varsに指定された配置先を確認する。サービス用ユーザーとファイル権限は、続くD12の構築手順で設定される。


In [ ]:
https_keystore_path = configuration['https_keystore_path']
https_truststore_path = configuration['https_truststore_path']
ca_bundle_path = '/etc/hadoop/conf/ca-certificates.pem'
(https_keystore_path, https_truststore_path, ca_bundle_path)


In [ ]:
!ansible {target_group} --become -m file -a 'path=/etc/hadoop/conf state=directory owner=root group=root mode=0755'


In [ ]:
!ansible {target_group} --become -m copy -a 'src={truststore_file} dest={https_truststore_path} owner=root group=root mode=0644'
!ansible {target_group} --become -m copy -a 'src={ca_certificate_file} dest={ca_bundle_path} owner=root group=root mode=0644'


In [ ]:
for address, hostname in certificate_hosts.items():
    !ansible {address} --become -m copy -a 'src={pki_dir}/{hostname}-keystore.jks dest={https_keystore_path} owner=root group=root mode=0600'


配置したCA証明書とファイルの所有者・権限を確認する。


In [ ]:
!ansible {target_group} -m shell -a 'openssl x509 -in {ca_bundle_path} -noout -text -fingerprint -sha256'
!ansible {target_group} -m shell -a 'ls -l {https_keystore_path} {https_truststore_path} {ca_bundle_path}'


# 後始末

配置済みのホスト秘密鍵を操作端末から削除する。CAの鍵と証明書は、今後の発行に使用するため保管する。


In [ ]:
for address, hostname in certificate_hosts.items():
    !rm "{pki_dir}/{hostname}-keystore.jks"


In [ ]:
del os.environ['HADOOP_CA_PASSWORD']
del os.environ['HADOOP_KEYSTORE_PASSWORD']
del os.environ['HADOOP_PRIVATE_KEY_PASSWORD']
del os.environ['HADOOP_TRUSTSTORE_PASSWORD']
